# 06 · Image Manipulation for Vision

**Goal:** change one image on purpose, compare the result, and decide whether a vision output should still be trusted.

We use the same public source image for every transformation so that the change is visible and reproducible.


## 1. Get one source image

The image below comes from the Ultralytics public example set. Download it once, then keep the original unchanged.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import matplotlib.pyplot as plt

IMAGE_URL = 'https://ultralytics.com/images/bus.jpg'
source_path = Path('bus.jpg')
if not source_path.exists():
    urlretrieve(IMAGE_URL, source_path)

original = Image.open(source_path).convert('RGB')
original.size


In [ ]:
plt.figure(figsize=(11, 7))
plt.imshow(original)
plt.title('Original input')
plt.axis('off');


## 2. Make controlled transformations

Each function changes a different kind of evidence. Keep the original as the comparison point.


In [ ]:
def crop_context(image):
    width, height = image.size
    return image.crop((width * 0.28, height * 0.08, width * 0.88, height * 0.92))

def remove_detail(image, scale=0.18):
    small = image.resize((int(image.width * scale), int(image.height * scale)))
    return small.resize(image.size)

def lower_light(image, factor=0.32):
    return ImageEnhance.Brightness(image).enhance(factor)

def soften_edges(image, radius=8):
    return image.filter(ImageFilter.GaussianBlur(radius=radius))

def remove_colour(image):
    return ImageOps.grayscale(image).convert('RGB')


In [ ]:
transforms = {
    'Crop: context': crop_context(original),
    'Resize: detail': remove_detail(original),
    'Low light: contrast': lower_light(original),
    'Blur: edges': soften_edges(original),
    'Grayscale: colour': remove_colour(original),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()
axes[0].imshow(original); axes[0].set_title('Original')
for ax, (name, image) in zip(axes[1:], transforms.items()):
    ax.imshow(image); ax.set_title(name)
for ax in axes: ax.axis('off')
plt.tight_layout()


## 3. Test a claim

Choose one transformation and write a prediction before you run it:

- What object or clue becomes harder to see?
- Would a detector have enough evidence to make the same claim?
- What would you record so another person could repeat your test?


In [ ]:
# Change one value and rerun. Record both the parameter and what you observe.
test = soften_edges(original, radius=14)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(original); axes[0].set_title('Original')
axes[1].imshow(test); axes[1].set_title('Test: Gaussian blur radius = 14')
for ax in axes: ax.axis('off')
plt.tight_layout()


## 4. Deliberately test failure

A clear image is an easy case. Test one difficult condition: low light, motion blur, a crop, a different angle, or an occlusion.

**Responsible-use prompt:** If this system makes a consequential decision, who checks the uncertain result, and what evidence do they need to see?
